# v4.5 — signal_rebuild.ipynb

Rebuilds the signal combination CSV using `dir_120` (11:15 outcome) as the
target variable instead of `dir_60` (60-min return used in v2).

**Run this once** before running `backtest.ipynb` in v4.5.
Output: `v45_reliable_signals.csv` (same schema as `v2/v2_reliable_signals.csv`).


In [1]:
import sys
from pathlib import Path
HERE   = Path(globals().get('__vsc_ipynb_file__', Path.cwd())).parent
V2_DIR = HERE.parent.parent / 'v2'
import pandas as pd
import numpy  as np
import warnings
from scipy.stats import binomtest
warnings.filterwarnings('ignore')

ALIGNED_CSV = V2_DIR / 'v2_aligned_dataset.csv'
SPOT_DIR    = V2_DIR / 'backtesting_2024_options' / '2024' / '2024Nifty'
KITE_CACHE  = HERE.parent.parent / 'kite_minute_cache'
OUT_CSV     = HERE / 'v45_reliable_signals.csv'

BASE_RATE   = 54.5   # % sessions where first-hour closes DOWN
MIN_N       = 40     # minimum occurrences to test a combo
MAX_PVAL    = 0.05   # significance threshold

print(f"Aligned dataset: {ALIGNED_CSV}")
print(f"Output: {OUT_CSV}")


Aligned dataset: c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\gap_trading\v2\v2_aligned_dataset.csv
Output: c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\gap_trading\v4\v4.5\v45_reliable_signals.csv


In [2]:
# ── Cell 2: Load aligned dataset and build dir_120 ───────────────────────────
df = pd.read_csv(ALIGNED_CSV)
print(f"Loaded aligned dataset: {df.shape}")
print(f"Columns: {df.columns.tolist()[:10]} ...")

# ── Load NIFTY 1-min spot from 2024Nifty CSVs ────────────────────────────────
from datetime import time as dtime
spot_1115 = {}   # date -> NIFTY close at 11:15
spot_0915 = {}   # date -> NIFTY open  at 09:15

for f in sorted(SPOT_DIR.glob('Nifty-2024*.csv')):
    try:
        s = pd.read_csv(f)
        s.columns = [c.strip().lower() for c in s.columns]
        s['dt'] = pd.to_datetime(s['datetime'])
        s['date'] = s['dt'].dt.date
        s['time'] = s['dt'].dt.time
        for d, grp in s.groupby('date'):
            r915 = grp[grp['time'] == dtime(9,15)]
            r1115 = grp[grp['time'] == dtime(11,15)]
            if not r915.empty:
                spot_0915[str(d)] = float(r915.iloc[0]['open'])
            if not r1115.empty:
                spot_1115[str(d)] = float(r1115.iloc[-1]['close'])
    except Exception as e:
        print(f"  [warn] {f.name}: {e}")

print(f"9:15 open  available for {len(spot_0915)} days")
print(f"11:15 close available for {len(spot_1115)} days")

# ── Compute ret_110 and dir_120 ───────────────────────────────────────────────
def _ret110(row):
    d = str(row.get('india_date', ''))[:10]
    o = spot_0915.get(d)
    c = spot_1115.get(d)
    if o and c and o > 0:
        return (c - o) / o
    return np.nan

df['ret_110'] = df.apply(_ret110, axis=1)
df['dir_120'] = df['ret_110'].apply(lambda x: -1.0 if x < 0 else (1.0 if x > 0 else np.nan))

valid = df['dir_120'].notna().sum()
down  = (df['dir_120'] == -1.0).sum()
print(f"\ndir_120 computed for {valid} / {len(df)} rows")
print(f"  DOWN (dir_120 = -1): {down} ({down/valid*100:.1f}%)")
print(f"  UP   (dir_120 = +1): {valid-down} ({(valid-down)/valid*100:.1f}%)")


Loaded aligned dataset: (740, 46)
Columns: ['india_date', 'india_open', 'ret_30', 'ret_60', 'dir_30', 'dir_60', 'gap_pct', 'gap_dir', 'prev_india_ret', 'SP500_ret'] ...
9:15 open  available for 245 days
11:15 close available for 245 days

dir_120 computed for 245 / 740 rows
  DOWN (dir_120 = -1): 132 (53.9%)
  UP   (dir_120 = +1): 113 (46.1%)


In [3]:
# ── Cell 3: Re-run signal combination analysis on dir_120 ──────────────────────
# Use same signal columns as v2 (already binarised in aligned dataset)
SIG_COLS = {
    'Gap Up':         lambda r: r.get('gap_pct', 0)  >  0.0015,
    'Gap Up Strong':  lambda r: r.get('gap_pct', 0)  >  0.0050,
    'Gap Down':       lambda r: r.get('gap_pct', 0)  < -0.0015,
    'Prev India UP':  lambda r: r.get('prev_india_ret', 0) > 0,
    'Prev India DOWN':lambda r: r.get('prev_india_ret', 0) < 0,
    'US UP':          lambda r: r.get('SP500_ret', 0)  > 0,
    'US DOWN':        lambda r: r.get('SP500_ret', 0)  < 0,
    'SGX UP':         lambda r: r.get('SGX_ret', 0)    > 0,
    'SGX DOWN':       lambda r: r.get('SGX_ret', 0)    < 0,
    'DAX UP':         lambda r: r.get('DAX_ret', 0)    > 0,
    'VIX Rising':     lambda r: r.get('VIX_US_ret', 0) > 0.03,
    'VIX Falling':    lambda r: r.get('VIX_US_ret', 0) < 0,
    'VIX Spike':      lambda r: r.get('VIX_US_ret', 0) > 0.05,
}

# Build binary signal matrix
sub = df[df['dir_120'].notna()].copy()
for name, fn in SIG_COLS.items():
    sub[name] = sub.apply(fn, axis=1).astype(bool)
sub['target_down'] = (sub['dir_120'] == -1.0)

print(f"Working dataset: {len(sub)} rows with dir_120 target")

# ── Test all 2-4 signal combinations ─────────────────────────────────────────
from itertools import combinations
sig_names = list(SIG_COLS.keys())
results = []

for level in [2, 3, 4]:
    for combo in combinations(sig_names, level):
        mask = sub[list(combo)].all(axis=1)
        n = mask.sum()
        if n < MIN_N: continue
        n_down = sub.loc[mask, 'target_down'].sum()
        p_down = n_down / n * 100
        p_up   = 100 - p_down
        freq   = n / len(sub) * 100

        # Binomial test: is P(DOWN) significantly different from base rate?
        direction = 'DOWN' if p_down > BASE_RATE else 'UP'
        test_k    = n_down if direction == 'DOWN' else (n - n_down)
        test_p    = BASE_RATE / 100 if direction == 'DOWN' else (1 - BASE_RATE / 100)
        result    = binomtest(test_k, n, test_p, alternative='greater')
        pval      = result.pvalue

        if pval < MAX_PVAL:
            edge = abs(p_down - BASE_RATE)
            # 95% CI
            ci_lo = (test_k - 1.96 * (n * test_p * (1-test_p)) ** 0.5) / n * 100
            ci_hi = (test_k + 1.96 * (n * test_p * (1-test_p)) ** 0.5) / n * 100
            sig_str = '***' if pval < 0.001 else ('**' if pval < 0.01 else '*')

            results.append({
                'N': n, 'Freq_pct': round(freq,1),
                'P_Down': round(p_down,1), 'P_Up': round(p_up,1),
                'Edge_pp': round(edge,1),
                'CI_lo': round(ci_lo,1), 'CI_hi': round(ci_hi,1),
                'p_val': round(pval,4), 'Sig': sig_str,
                'Direction': direction, 'Verdict': 'RELIABLE',
                'Level': level,
                'Signal': ' + '.join(combo),
            })

results_df = pd.DataFrame(results).sort_values('Edge_pp', ascending=False).reset_index(drop=True)
print(f"\nFound {len(results_df)} reliable combos (dir_120 target, p<{MAX_PVAL}, N>={MIN_N})")
results_df.head(15)


Working dataset: 245 rows with dir_120 target

Found 15 reliable combos (dir_120 target, p<0.05, N>=40)


,N,Freq_pct,P_Down,P_Up,Edge_pp,CI_lo,CI_hi,p_val,Sig,Direction,Verdict,Level,Signal
0,57,23.3,33.3,66.7,21.2,53.7,79.6,0.0010,**,UP,RELIABLE,2,Prev India DOWN + US DOWN
1,45,18.4,33.3,66.7,21.2,52.1,81.2,0.0034,**,UP,RELIABLE,3,Prev India DOWN + US DOWN + SGX DOWN
2,56,22.9,35.7,64.3,18.8,51.2,77.3,0.0036,**,UP,RELIABLE,2,Prev India DOWN + SGX DOWN
3,62,25.3,69.4,30.6,14.9,57.0,81.8,0.0122,*,DOWN,RELIABLE,3,Gap Up + US UP + DAX UP
4,74,30.2,68.9,31.1,14.4,57.6,80.3,0.0081,**,DOWN,RELIABLE,3,US UP + SGX UP + DAX UP
5,90,36.7,68.9,31.1,14.4,58.6,79.2,0.0038,**,DOWN,RELIABLE,2,US UP + DAX UP
6,48,19.6,68.8,31.2,14.2,54.7,82.8,0.0317,*,DOWN,RELIABLE,2,Prev India DOWN + US UP
7,51,20.8,41.2,58.8,13.3,45.2,72.5,0.0386,*,UP,RELIABLE,2,US DOWN + VIX Rising
8,55,22.4,67.3,32.7,12.8,54.1,80.4,0.0374,*,DOWN,RELIABLE,4,Gap Up + US UP + SGX UP + DAX UP
9,76,31.0,42.1,57.9,12.4,46.7,69.1,0.0201,*,UP,RELIABLE,2,US DOWN + SGX DOWN


In [4]:
# ── Cell 4: Compare v2 vs v4.5 top-10 bearish combos ──────────────────────────
v2_csv = V2_DIR / 'v2_reliable_signals.csv'
v2_df  = pd.read_csv(v2_csv)
v2_bear = v2_df[v2_df['P_Down'] > BASE_RATE].sort_values('Edge_pp', ascending=False).head(10)
v45_bear = results_df[results_df['Direction'] == 'DOWN'].head(10)

print("=" * 72)
print("  v2 TOP-10 BEARISH (trained on dir_60)")
print("=" * 72)
for i, r in v2_bear.iterrows():
    print(f"  {r['P_Down']:>5.1f}%  N={r['N']:>4}  {r['Signal']}")

print()
print("=" * 72)
print("  v4.5 TOP-10 BEARISH (trained on dir_120)")
print("=" * 72)
for i, r in v45_bear.iterrows():
    print(f"  {r['P_Down']:>5.1f}%  N={r['N']:>4}  {r['Signal']}")

# ── Save ──────────────────────────────────────────────────────────────────────
results_df.to_csv(OUT_CSV, index=False)
print(f"\nSaved {len(results_df)} combos to {OUT_CSV}")


  v2 TOP-10 BEARISH (trained on dir_60)
   74.6%  N=  67  Gap Up + Prev India DOWN + US UP + SGX UP
   74.1%  N=  54  Gap Up + Prev India DOWN + SGX UP + DAX UP
   72.7%  N=  44  Gap Up + Prev India DOWN + SGX UP + VIX Falling
   72.6%  N=  73  Gap Up + Prev India DOWN + SGX UP
   70.7%  N=  58  Gap Up + Prev India DOWN + US UP + DAX UP
   70.0%  N= 100  Gap Up + SGX UP + DAX UP + VIX Falling
   68.5%  N= 111  Gap Up + DAX UP + VIX Falling
   68.3%  N= 126  Gap Up + SGX UP + VIX Falling
   68.2%  N= 154  Gap Up + SGX UP + DAX UP
   68.1%  N= 138  Gap Up + US UP + SGX UP + DAX UP

  v4.5 TOP-10 BEARISH (trained on dir_120)
   69.4%  N=  62  Gap Up + US UP + DAX UP
   68.9%  N=  74  US UP + SGX UP + DAX UP
   68.9%  N=  90  US UP + DAX UP
   68.8%  N=  48  Prev India DOWN + US UP
   67.3%  N=  55  Gap Up + US UP + SGX UP + DAX UP
   66.3%  N= 104  US UP + SGX UP
   66.3%  N=  86  SGX UP + DAX UP
   65.7%  N=  67  Gap Up + US UP + SGX UP
   65.4%  N=  81  Gap Up + US UP
   65.3%  N=  75  